# Wholesale Partner CX case study: analysis notebook
**Lynn Le · Bachelor of Business Analytics, Deakin University**

This notebook backs up every number in my case study deck on the Aussie Broadband Group's Wholesale business (Symbio).
It runs top to bottom in Google Colab.

| Section | What it does | Tools |
|---|---|---|
| 1 | Recalculates the headline numbers from the public sources (Annual Report, TIO) | pandas |
| 2 | Charts the Ombudsman complaint trend | matplotlib |
| 3 | Tags Trustpilot reviews into 8 CX themes with an AI model, scores the tone, and charts complaints against praise | Hugging Face `transformers` (zero-shot AI), VADER sentiment |
| 4 | Writes the partner relationship survey as a file you can import straight into Qualtrics | Qualtrics Advanced Format |
| 5 | NPS, CSAT and effort-score functions for the partner scorecard | Python |

All data is public. Not affiliated with Aussie Broadband or Symbio.

## 0 · Setup

In [ ]:
!pip -q install vaderSentiment transformers

import re, math
import pandas as pd
import matplotlib.pyplot as plt

# Deck colours, so charts match the slides
INK, MUTED, BLUE, ORANGE, TEAL = "#10202B", "#5F6F79", "#2A78D6", "#EB6834", "#0A6B60"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

## 1 · Headline numbers (slide 2)
Every input is typed in from its source, with the page number, so anyone can check it.

In [ ]:
# Aussie Broadband Group, Annual Report 2026 (FY26 = year to 30 June 2026)
# https://wcsecure.weblink.com.au/pdf/ABB/03124593.pdf
wholesale_revenue_m      = 297.3     # p.32, up 9.4%
wholesale_bb_connections = 336_000   # p.32
wholesale_bb_growth      = 5.46      # p.32, "up 546%"
migrated_services        = 269_000   # p.6, More/Tangerine services on the network by 30 June 2026
partners                 = 1_000     # p.30, "well over a thousand wholesale partners"
published_partner_scores = 0         # no NPS or partner satisfaction figure anywhere in the report (p.22 mentions NPS only in words)

# Telecommunications Industry Ombudsman (TIO), Complaints Data Insights
abb_jul_sep_2025      = 203      # Q1 FY26 report, Table 6, p.16
abb_oct_dec_2025      = 302      # Q3 FY26 report, Table 2, p.7 (revised; first published as 303)
industry_jul_sep_2025 = 13_547   # Q1 FY26 report, p.4
industry_oct_dec_2025 = 14_017   # Q2 FY26 report, p.4

connections_before   = wholesale_bb_connections / (1 + wholesale_bb_growth)
share_from_migration = migrated_services / wholesale_bb_connections
abb_qoq              = abb_oct_dec_2025 / abb_jul_sep_2025 - 1
industry_qoq         = industry_oct_dec_2025 / industry_jul_sep_2025 - 1
abb_share_of_all     = abb_oct_dec_2025 / industry_oct_dec_2025

headline = pd.DataFrame([
    ["Wholesale broadband growth",          f"{wholesale_bb_growth:.0%}", f"from ~{connections_before/1000:.0f}k to {wholesale_bb_connections/1000:.0f}k connections"],
    ["Share from one migration",            f"{share_from_migration:.0%}", f"{migrated_services/1000:.0f}k of {wholesale_bb_connections/1000:.0f}k"],
    ["Group Ombudsman complaints, QoQ",     f"{abb_qoq:+.0%}",            f"{abb_jul_sep_2025} → {abb_oct_dec_2025} (Oct–Dec 2025)"],
    ["Industry Ombudsman complaints, QoQ",  f"{industry_qoq:+.1%}",       f"{industry_jul_sep_2025:,} → {industry_oct_dec_2025:,}"],
    ["Group share of all TIO complaints",   f"{abb_share_of_all:.1%}",     "Oct–Dec 2025"],
    ["Published partner satisfaction scores", published_partner_scores,   f"across {partners:,}+ partners"],
], columns=["Metric", "Value", "Working"])
headline

## 2 · Ombudsman complaints by quarter (slide 4)
The TIO doesn't publish an April–June quarterly report, so those two quarters are blank.
Where the TIO later revised a figure, I use the revised one.

In [ ]:
tio = pd.DataFrame([
    # quarter,  complaints, source
    ("Q1 '24", 257,  "Jan–Mar 2025 report, Table 3 p.11 (prior-year column; first published as 261)"),
    ("Q2 '24", None, "No Apr–Jun quarterly report"),
    ("Q3 '24", 231,  "Jul–Sep 2024 report, Table 3 p.6"),
    ("Q4 '24", 248,  "Oct–Dec 2024 report, Table 3 p.7"),
    ("Q1 '25", 242,  "Jan–Mar 2025 report, Table 3 p.11"),
    ("Q2 '25", None, "No Apr–Jun quarterly report"),
    ("Q3 '25", 203,  "Jul–Sep 2025 report, Table 6 p.16"),
    ("Q4 '25", 302,  "Jan–Mar 2026 report, Table 2 p.7 (revised; first published as 303)"),
    ("Q1 '26", 254,  "Jan–Mar 2026 report, Table 2 p.7"),
], columns=["quarter", "complaints", "source"])

tio["qoq_change"] = tio["complaints"].pct_change(fill_method=None)
display(tio)

fig, ax = plt.subplots(figsize=(10, 4.5))
colours = [ORANGE if q == "Q4 '25" else BLUE for q in tio["quarter"]]
ax.bar(tio["quarter"], tio["complaints"].fillna(0), color=colours, width=0.55)
for i, v in enumerate(tio["complaints"]):
    ax.text(i, (v or 0) + 5, "n/a" if pd.isna(v) else int(v), ha="center", color=MUTED if pd.isna(v) else INK)
ax.axvspan(7.6, 8.4, color=ORANGE, alpha=0.08)
ax.text(8, 330, "Migration\nJan–Jun '26", ha="center", color=ORANGE, fontsize=9)
ax.set_ylim(0, 360); ax.set_yticks([])
ax.set_title(f"Complaints spiked {abb_qoq:+.0%} in late 2025 (industry {industry_qoq:+.1%}), then fell during the migration",
             loc="left", color=INK, fontweight="bold")
plt.tight_layout(); plt.show()

### Review scores across sites (slide 3)
Live pages change, so record the date you checked them.

In [ ]:
ratings = pd.DataFrame([
    ("ProductReview.com.au", 4.4,  "out of 5", "10,279 reviews when collected"),
    ("Trustpilot",           1.8,  "out of 5", "317 reviews when collected"),
    ("WhistleOut Awards 2026, customer service", 4.19, "out of 5", "min. 50 responses per provider"),
    ("Finder Awards 2026, overall",              4.10, "out of 5", "survey of recent customers"),
    ("CHOICE 2026, satisfied",                   0.80, "share",    "n = 110 Aussie Broadband customers"),
], columns=["source", "score", "scale", "note"])
ratings

## 3 · AI theme coding of Trustpilot reviews (slide 5)

**How it works**
1. Each review is split into sentences.
2. A zero-shot AI model (`facebook/bart-large-mnli`) reads each sentence and scores it against the 8 CX themes. It needs no training data, only the theme descriptions.
3. VADER scores the tone of each sentence as positive or negative. If VADER is neutral, the star rating breaks the tie.
4. Each review counts once per theme and tone, so one review can mention several themes.
5. Every tag is exported to a CSV so I can check it by hand. The AI does the first pass and a person makes the final call.

**Input:** a CSV called `trustpilot_reviews.csv` with the columns `date`, `rating` (1–5) and `text`.
Tip: in Colab, go to *Runtime → Change runtime type → T4 GPU* to make this cell run faster.

In [ ]:
# Upload your reviews file (or put it in the Colab file panel)
import os
if not os.path.exists("trustpilot_reviews.csv"):
    try:
        from google.colab import files
        files.upload()   # pick trustpilot_reviews.csv
    except ImportError:
        pass

reviews = pd.read_csv("trustpilot_reviews.csv")
reviews = reviews.dropna(subset=["text"]).reset_index(drop=True)
reviews["review_id"] = reviews.index + 1
print(f"{len(reviews)} reviews loaded")
print(f"One-star share: {(reviews['rating'] == 1).mean():.0%}")
reviews.head()

In [ ]:
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

MODEL = "facebook/bart-large-mnli"   # zero-shot classifier
THRESHOLD = 0.60                     # minimum AI confidence to tag a theme

# Theme name -> plain-English description the AI model matches against
THEMES = {
    "Support":                  "customer support, phone wait times, callbacks or escalations",
    "Billing and pricing":      "billing, charges, price rises or value for money",
    "Speed and performance":    "internet speed or performance",
    "Outages and reliability":  "outages, dropouts or reliability",
    "Communication":            "communication, updates or notifications from the company",
    "Onboarding and switching": "signing up, connecting, switching or moving a service",
    "Mobile and new products":  "mobile plans or new products",
    "App and self-service":     "the app, website or self-service tools",
}
labels = list(THEMES.values())
label_to_theme = {v: k for k, v in THEMES.items()}

classifier = pipeline("zero-shot-classification", model=MODEL)
vader = SentimentIntensityAnalyzer()

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|\n+", str(text))
    return [p.strip() for p in parts if len(p.strip()) > 3]

def tone(sentence, stars):
    score = vader.polarity_scores(sentence)["compound"]
    if score >= 0.05:  return "positive"
    if score <= -0.05: return "negative"
    return "negative" if stars <= 2 else "positive" if stars >= 4 else None

rows = []
for r in reviews.itertuples():
    sentences = split_sentences(r.text)
    if not sentences:
        continue
    results = classifier(sentences, labels, multi_label=True,
                         hypothesis_template="This review is about {}.")
    if isinstance(results, dict):
        results = [results]
    for sentence, res in zip(sentences, results):
        for label, score in zip(res["labels"], res["scores"]):
            if score >= THRESHOLD:
                rows.append({"review_id": r.review_id, "rating": r.rating, "sentence": sentence,
                             "theme": label_to_theme[label], "ai_confidence": round(score, 2),
                             "tone": tone(sentence, r.rating)})

tags = pd.DataFrame(rows).dropna(subset=["tone"])
tags["checked_by_hand"] = ""   # fill in: ok / wrong theme / wrong tone
tags.to_csv("review_tags_for_checking.csv", index=False)
print(f"{len(tags)} theme mentions tagged across {tags['review_id'].nunique()} reviews")
tags.head(10)

In [ ]:
# Count each review once per theme and tone
counts = (tags.drop_duplicates(["review_id", "theme", "tone"])
              .pivot_table(index="theme", columns="tone", values="review_id", aggfunc="count", fill_value=0)
              .reindex(columns=["negative", "positive"], fill_value=0)
              .reindex(THEMES.keys(), fill_value=0)
              .sort_values("negative"))
display(counts.sort_values("negative", ascending=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(counts.index, -counts["negative"], color=ORANGE)
ax.barh(counts.index, counts["positive"], color=BLUE)
for i, (n, p) in enumerate(zip(counts["negative"], counts["positive"])):
    ax.text(-n - 1, i, n, va="center", ha="right"); ax.text(p + 1, i, p, va="center")
ax.axvline(0, color=MUTED, lw=1); ax.set_xticks([])
ax.set_title(f"What people complain about (left) and praise (right), {len(reviews)} Trustpilot reviews",
             loc="left", color=INK, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Optional: compare the AI's counts with my hand-coded counts from the deck
hand_coded = pd.DataFrame({
    "negative": {"Support": 56, "Billing and pricing": 29, "Speed and performance": 25, "Outages and reliability": 24,
                 "Communication": 13, "Onboarding and switching": 11, "Mobile and new products": 10, "App and self-service": 4},
    "positive": {"Support": 19, "Billing and pricing": 2, "Speed and performance": 3, "Outages and reliability": 6,
                 "Communication": 3, "Onboarding and switching": 4, "Mobile and new products": 1, "App and self-service": 1},
})
comparison = hand_coded.join(counts, rsuffix="_ai")
comparison["neg_gap"] = comparison["negative_ai"] - comparison["negative"]
comparison["pos_gap"] = comparison["positive_ai"] - comparison["positive"]
comparison.sort_values("negative", ascending=False)

## 4 · Partner relationship survey as a Qualtrics import file (slide 8)
This writes the survey and the three pulses in Qualtrics *Advanced Format*.
In Qualtrics: **Create project → Survey → Tools → Import/Export → Import survey**, then choose the `.txt` file.

In [ ]:
touchpoints = ["Onboarding", "Network performance", "Fault fixing", "Billing",
               "Notice of changes", "Partner Manager", "Partner portal", "API"]
sat_scale = ["Very dissatisfied", "Dissatisfied", "Neutral", "Satisfied", "Very satisfied"]

def mc(qid, text, choices, horizontal=True):
    kind = "Horizontal" if horizontal else "Vertical"
    return f"[[Question:MC:SingleAnswer:{kind}]]\n[[ID:{qid}]]\n{text}\n[[Choices]]\n" + "\n".join(map(str, choices)) + "\n"

def matrix(qid, text, rows, scale):
    return (f"[[Question:Matrix]]\n[[ID:{qid}]]\n{text}\n[[Choices]]\n" + "\n".join(rows) +
            "\n[[Answers]]\n" + "\n".join(scale) + "\n")

def essay(qid, text):
    return f"[[Question:TE:Essay]]\n[[ID:{qid}]]\n{text}\n"

blocks = {
    "Relationship survey (every 6 months)": [
        mc("NPS", "How likely are you to recommend us as a wholesale partner to a colleague? (0 = not at all likely, 10 = extremely likely)", range(0, 11)),
        essay("NPS_why", "What is the main reason for your score?"),
        matrix("SAT", "How satisfied are you with each of these?", touchpoints, sat_scale),
        mc("CES", "How easy is it to get a fault fixed for your customer? (1 = very difficult, 7 = very easy)", range(1, 8)),
        mc("BRAND", "In the last 6 months, did an issue on our side affect how your customers see your brand?", ["Yes", "No", "Not sure"], horizontal=False),
    ],
    "Onboarding pulse (at go-live)": [
        mc("ONB_EASE", "How easy was it to go live with us? (1 = very difficult, 7 = very easy)", range(1, 8)),
        essay("ONB_SLOW", "What slowed you down?"),
    ],
    "Migration pulse (day 7 and day 30)": [
        mc("MIG_DISRUPT", "Were any of your customers' services disrupted during the migration?", ["Yes", "No", "Not sure"], horizontal=False),
        mc("MIG_INFORMED", "How well were you kept informed? (1 = not at all, 5 = completely)", range(1, 6)),
    ],
    "Ticket CSAT (when a ticket closes)": [
        mc("CSAT", "How satisfied are you with how this ticket was handled?", sat_scale),
    ],
}

lines = ["[[AdvancedFormat]]", ""]
for block, questions in blocks.items():
    lines += [f"[[Block:{block}]]", ""]
    for q in questions:
        lines += [q, "[[PageBreak]]", ""]
qualtrics_txt = "\n".join(lines)

with open("partner_cx_survey_qualtrics.txt", "w") as f:
    f.write(qualtrics_txt)
print(qualtrics_txt[:900], "...")

try:
    from google.colab import files
    files.download("partner_cx_survey_qualtrics.txt")
except ImportError:
    pass

## 5 · Scorecard functions (slide 9)
How the monthly partner CX scorecard would be calculated once real responses come in.

In [ ]:
def nps(scores):
    """Net Promoter Score: % promoters (9–10) minus % detractors (0–6)."""
    s = pd.Series(scores)
    return round(100 * ((s >= 9).mean() - (s <= 6).mean()))

def csat(scores, top=(4, 5)):
    """CSAT: % of answers in the top two boxes of a 1–5 scale."""
    return round(100 * pd.Series(scores).isin(top).mean())

def effort(scores):
    """Customer Effort Score: average on a 1–7 scale (higher = easier)."""
    return round(pd.Series(scores).mean(), 1)

def detractor_callbacks(responses):
    """Partners who scored 0–6 and need a call within 5 business days."""
    return responses[responses["nps"] <= 6][["partner", "nps"]]

# Example with made-up responses, just to show the functions working
example = pd.DataFrame({"partner": list("ABCDEFGH"), "nps": [10, 9, 8, 7, 6, 9, 3, 10]})
print("Partner NPS:", nps(example["nps"]))
detractor_callbacks(example)